# Stage 1E: QA Generation (Placeholder)

| Version | Date | Author | Description |
| --- | --- | --- | --- |
| 1.0.0 | 2026-01-28 | That Le | Placeholder for QA generation |

---

## Status: PENDING

This notebook will generate QA pairs after:
1. Stage 3 (OCR) extracts text from charts
2. Stage 4 (SLM) provides reasoning capabilities

### Current Pipeline Status

| Stage | Status | Output |
| --- | --- | --- |
| 01a | ✅ Done | 9,508 PDFs |
| 01b | ✅ Done | 198,887 images |
| 01c | ✅ Done | 50,865 detected charts |
| 01d | ✅ Done | 50,865 classified by type |
| 01e | 🔄 Pending | QA pairs (requires Stage 3-4) |

### Next Steps

1. **Review uncertain charts** (10,326 images)
2. **Re-train models** with expanded dataset
3. **Implement Stage 3** (OCR extraction)
4. **Implement Stage 4** (SLM reasoning)
5. **Generate QA pairs** using SLM

In [ ]:
# ============================================================================
# CURRENT DATA SUMMARY
# ============================================================================
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
classified_dir = PROJECT_ROOT / "data" / "academic_dataset" / "classified_charts"

print("=" * 60)
print("CLASSIFIED CHARTS SUMMARY")
print("=" * 60)

total = 0
by_type = {}

for d in sorted(classified_dir.iterdir()):
    if d.is_dir():
        count = len(list(d.glob("*.png"))) + len(list(d.glob("*.jpg")))
        if count > 0:
            by_type[d.name] = count
            total += count

for name, count in sorted(by_type.items(), key=lambda x: -x[1]):
    pct = count / total * 100
    print(f"  {name:15s}: {count:6,} ({pct:5.1f}%)")

print("-" * 60)
print(f"  {'TOTAL':15s}: {total:6,}")
print("=" * 60)

In [ ]:
# ============================================================================
# SAMPLE IMAGES BY TYPE
# ============================================================================
import random
from PIL import Image
import matplotlib.pyplot as plt

# Select one random image from each type
chart_types = [d.name for d in classified_dir.iterdir() if d.is_dir() and d.name != "uncertain"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, ct in zip(axes, chart_types):
    ct_dir = classified_dir / ct
    images = list(ct_dir.glob("*.png"))
    if images:
        img_path = random.choice(images)
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(f"{ct}\n({by_type.get(ct, 0):,})", fontsize=11, fontweight="bold")
    ax.axis("off")

plt.suptitle("Sample Charts by Type", fontsize=14)
plt.tight_layout()
plt.show()

---

## QA Generation Strategy (Future)

When Stage 3-4 are ready:

### QA Categories

| Category | Example Question | Source |
| --- | --- | --- |
| **Type** | "What type of chart is this?" | ResNet-18 |
| **Structural** | "What is the chart title?" | OCR (Stage 3) |
| **Value** | "What is the value for X?" | OCR + Geometric (Stage 3) |
| **Comparison** | "Which has the highest value?" | SLM reasoning (Stage 4) |
| **Trend** | "Is there an increasing trend?" | SLM reasoning (Stage 4) |

### Output Format

```json
{
  "image": "chart_001.png",
  "chart_type": "bar",
  "qa_pairs": [
    {
      "question": "What type of chart is this?",
      "answer": "This is a bar chart.",
      "category": "type"
    },
    {
      "question": "What is the title?",
      "answer": "Sales by Region",
      "category": "structural"
    }
  ]
}
```

In [ ]:
# ============================================================================
# PLACEHOLDER: Basic QA Generation (Type only)
# ============================================================================
import json
from datetime import datetime

GENERATE_BASIC_QA = False  # Set True to generate type-only QA

if GENERATE_BASIC_QA:
    qa_data = []
    
    for ct in chart_types:
        ct_dir = classified_dir / ct
        for img_path in ct_dir.glob("*.png"):
            qa_data.append({
                "image": img_path.name,
                "image_path": str(img_path.relative_to(PROJECT_ROOT)),
                "chart_type": ct,
                "qa_pairs": [
                    {
                        "question": "What type of chart is this?",
                        "answer": f"This is a {ct} chart.",
                        "category": "type"
                    }
                ]
            })
    
    # Save
    output_dir = PROJECT_ROOT / "data" / "academic_dataset" / "chart_qa"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    output_file = output_dir / f"qa_basic_{datetime.now().strftime('%Y%m%d')}.json"
    with open(output_file, "w") as f:
        json.dump(qa_data, f, indent=2)
    
    print(f"Generated {len(qa_data):,} basic QA entries")
    print(f"Saved to: {output_file}")
else:
    print("[SKIPPED] Set GENERATE_BASIC_QA = True to generate")

---

## Summary

**Data Factory Pipeline Complete (01a-01d)**

```
ArXiv PDFs (9,508)
    |
    v
Raw Images (198,887)
    |
    v  [YOLO]
Detected Charts (50,865)
    |
    v  [ResNet-18]
Classified Charts:
  - line:      16,154
  - bar:        8,587
  - scatter:    7,615
  - heatmap:    4,227
  - pie:        1,299
  - box:        1,051
  - histogram:    882
  - area:         724
  - uncertain: 10,326
```

**Next: Core Pipeline (Stage 2-5)**